<a href="https://colab.research.google.com/github/tlmartiner/Integracion_de_datos_y_prospectiva/blob/main/Reto_3_Integraci%C3%B3n_Multidimensional.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Integración Multidimensional - K-Medoids**

Una empresa internacional de e-commerce tiene seis bloques físicos destinados para la repartición de sus productos (A,B,C,D,E,F), sin embargo, por problemas de presupuesto, la compañía requiere reducir el número de bloques a solo tres (A,B,C). De acuerdo con lo anterior, la empresa quiere reorganizar la información para los nuevos bloques, mediante la utilización de un proceso de integración de datos multidimensional.

Tomando como variable de referencia los bloque dentinados para la repartición a(Warehouse_block) y como variables de entrada Customer_Care_Calls, Cost_of_the_product, Discount_offered, Reached.on.time_Y.N, ID y Weight_in_gms. Para esta integración se tomará como referencia el modelo clsuterización K-Medoids, el cual lleva a cabo una integración multidimensional (varias variables) y dinámica de los datos (integrar dato a dato). La base de datos para el desarrollo de este análisis se puede encontrar en: https://www.kaggle.com/datasets/prachi13/customer-analytics?resource=download

# **Descripción de la base de datos**

La base de datos cuenta con 12 variables de las cuales 8 son cuantitativas y 4 cualitativas. Las variables son las siguintes:

- **ID:** ID Number of Customers.
- **Warehouse block:** The Company have big Warehouse which is divided in to block such as A,B,C,D,E.
- **Mode of shipment:** The Company Ships the products in multiple way such as Ship, Flight and Road.
- **Customer care calls:** The number of calls made from enquiry for enquiry of the shipment.
- **Customer rating:** The company has rated from every customer. 1 is the lowest (Worst), 5 is the highest (Best).
- **Cost of the product:** Cost of the Product in US Dollars.
- **Prior purchases:** The Number of Prior Purchase.
- **Product importance:** The company has categorized the product in the various parameter such as low, medium, high.
- **Gender:** Male and Female.
- **Discount offered:** Discount offered on that specific product.
- **Weight in gms:** It is the weight in grams.
- **Reached on time:** It is the target variable, where 1 Indicates that the product has NOT reached on time and 0 indicates it has reached on time.

**Se importan librerias y base de datos**

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import skew, kurtosis
import random as rnd

In [2]:
nxl = '/content/Train.csv'
XDB = pd.read_csv(nxl)
XDB = XDB.dropna()

**Se sacan las variables numericas y se guardan en una variable**

In [3]:
vars_num = ['Customer_care_calls', 'Cost_of_the_Product', 'Discount_offered','Reached.on.Time_Y.N', 'Weight_in_gms']

**Se inicia la caracterización por bloque y se sacan metricas como la media, desviación estandar, coeficiente de asimetría y curtosis**

In [4]:
filas = np.where(XDB['Warehouse_block'].isin(['A','B','C']))
XDB_ref = XDB.iloc[filas[0], :]
XD = np.array(XDB_ref[vars_num])
yd=np.array(XDB[['Warehouse_block']])

In [8]:
#Caracterización inicial de A,B,C,D,E,F
def caracterizacion(df, var):
    # Check if the DataFrame subset for the variable is not empty before calculating
    if not df[var].dropna().empty:
        return {
            'media': df[var].mean(),
            'desviacion': df[var].std(),
            'asimetria': skew(df[var].dropna()),
            'curtosis': kurtosis(df[var].dropna())
        }
    else:
        # Return NaN or a specific indicator if the subset is empty
        return {
            'media': np.nan,
            'desviacion': np.nan,
            'asimetria': np.nan,
            'curtosis': np.nan
        }


print("Caracterización inicial por bloque:")
caracterizacion_data = []
for b in ['A','B','C','D','E','F']:
    sub = XDB[XDB['Warehouse_block']==b]
    # Add a check here to see if the subset DataFrame is empty
    if not sub.empty:
        for v in vars_num:
            char_results = caracterizacion(sub, v)
            char_results['Bloque'] = b
            char_results['Variable'] = v
            caracterizacion_data.append(char_results)
    else:
        print(f"Warning: No data found for Bloque '{b}' after dropping NaNs.")
        # Optionally, append rows with NaN metrics for this block and all variables
        # for v in vars_num:
        #     char_results = {'media': np.nan, 'desviacion': np.nan, 'asimetria': np.nan, 'curtosis': np.nan, 'Bloque': b, 'Variable': v}
        #     caracterizacion_data.append(char_results)


caracterizacion_df = pd.DataFrame(caracterizacion_data)
display(caracterizacion_df)

Caracterización inicial por bloque:


,media,desviacion,asimetria,curtosis,Bloque,Variable
0,4.038189,1.138673,0.411159,-0.320635,A,Customer_care_calls
1,208.767594,48.623021,-0.133272,-0.999503,A,Cost_of_the_Product
2,13.222586,16.214130,1.851199,2.182510,A,Discount_offered
3,0.586470,0.492601,-0.351172,-1.876678,A,Reached.on.Time_Y.N
4,3615.448991,1646.419545,-0.229296,-1.486960,A,Weight_in_gms
5,4.020185,1.123939,0.350274,-0.300418,B,Customer_care_calls
6,212.159302,46.980005,-0.231442,-0.859113,B,Cost_of_the_Product
7,13.187125,15.818064,1.822052,2.144065,B,Discount_offered
8,0.602291,0.489558,-0.418006,-1.825271,B,Reached.on.Time_Y.N
9,3635.701037,1642.093072,-0.258317,-1.449169,B,Weight_in_gms


**Se escoge la cantidad de bloques que se quieren generar**

In [9]:
np.random.seed(42)
Xmin = np.min(XD, axis=0)
Xmax = np.max(XD, axis=0)
print("Los valores minimos son:",Xmin)
print("Los valores máximos son:",Xmax)

import random as rnd   #esta función genera números entre 0 y 1

XC = np.zeros((3, len(vars_num)))
for j in range(3):
    XC[j,] = Xmin + (Xmax - Xmin)*np.random.rand(len(vars_num))
    XC[j,] = XD[j,]   # usar los primeros registros como centroides

fhat=np.zeros((len(XD),1))   #Esta variables nos determina el número de individuos por Bloque

for k in range(len(XD)):
  VP=np.exp(-0.5*(np.mean(((XC[:,]-XD[k,])/XC[:,])**2,axis=1)))
  nc=np.argmax(VP)
  fhat[k,]=int(nc)                  #Sabemos que dato pertenece a que cluster
  XC[nc,]=(XC[nc,]+XD[k,])/2   #Este es el K-medoids, a que cluster pertenece un dato

# Define yd from the original XDB DataFrame
yd = np.array(XDB_ref['Reached.on.Time_Y.N'])

fhat2=np.zeros((3,2))
for j in range(3):
  npx=len(np.where(fhat[:,]==j)[0])
  print("El numero de productos del bloque ",j," es ",npx)

  filas=np.where(fhat[:,]==j)[0]      #Filas de las personas Bloque tal
  fhat2[j,1]=len(np.where(yd[filas,]==1)[0])     #De estas personas cuantas sufriran ataque
  fhat2[j,0]=len(np.where(yd[filas,]==0)[0])     #De estas personas cuantas no sufriran
  # Avoid division by zero
  if np.sum(fhat2[j,]) > 0:
    fhat2[j,]=fhat2[j,]/np.sum(fhat2[j,])


XCT=np.column_stack((XC,fhat2))

dfXC=pd.DataFrame(XCT)
dfXC.columns=vars_num + ['Not_Reached', 'Reached']
dfXC.index=['Bloque A','Bloque B','Bloque C']
display(dfXC)

Los valores minimos son: [   2   96    1    0 1001]
Los valores máximos son: [   7  310   65    1 7401]
El numero de productos del bloque  0  es  3324
El numero de productos del bloque  1  es  571
El numero de productos del bloque  2  es  1604


,Customer_care_calls,Cost_of_the_Product,Discount_offered,Reached.on.Time_Y.N,Weight_in_gms,Not_Reached,Reached
Bloque A,4.603845,237.925525,4.158527,1.387957e-01,1248.333660,0.485860,0.514140
Bloque B,4.143689,235.302123,1.000000,8.935073e-93,1525.890719,0.698774,0.301226
Bloque C,3.796285,189.797589,1.408393,1.519291e-64,4525.042419,0.132170,0.867830


**Se comienza el proceso de integración de bloques**

In [10]:
nxl='/content/Train.csv'
XDB2=pd.read_csv(nxl)

filas2 = np.where(XDB2['Warehouse_block'].isin(['C','D','E','F']))
XDB2 = XDB.iloc[filas2[0], :]
XD2 = np.array(XDB2[vars_num])

In [11]:
XC2=np.copy(XC)

import pandas as pd
from scipy.stats import skew, kurtosis

vars_num = ['Customer_care_calls', 'Cost_of_the_Product',
            'Discount_offered', 'Reached.on.Time_Y.N', 'Weight_in_gms']

# CONVERTIR XC2 A DATAFRAME

if XC2.ndim == 1:
    XC2 = XC2.reshape(-1, len(vars_num))
has_label_col = XC2.shape[1] > len(vars_num)

XC2_df = pd.DataFrame(XC2[:, :len(vars_num)], columns=vars_num)

if has_label_col:
    XC2_df['Bloque'] = XC2[:, -1].astype(int)  # ajusta a int si viene float


for k in range(len(XD2)):
  VP2=np.exp(-0.5*(np.mean(((XC-XD2[k,])/XC)**2,axis=1)))
  VP2Ing=np.exp(-0.5*((((XC[:,2]-XD2[k,2])/XC[:,2])**2)))

  if np.max(VP2Ing)>0.95:
    nc2=np.argmax(VP2)
    print("El producto ",k,"pertenence al bloque ",nc2)
    print("Porcentaje de ser reasignado es:",fhat2[nc2,1])

    XC2[nc2,]=(XC2[nc2,]+XD2[k,])/2

#Los clusters originales son:
dfXC=pd.DataFrame(XC)
dfXC.columns=vars_num
display(dfXC)

#Estos son clusters o bloques modificados
dfXC2=pd.DataFrame(XC2)
dfXC2.columns=vars_num
display(dfXC2)

#Evaluamos los cambios porcentuales de cada una de las variables en los clusters
print("Los cambios porcentuales en cada una de las variables son:\n")
pd.DataFrame(np.abs((XC-XC2)/XC))

El producto  4 pertenence al bloque  0
Porcentaje de ser reasignado es: 0.5141395908543923
El producto  8 pertenence al bloque  0
Porcentaje de ser reasignado es: 0.5141395908543923
El producto  78 pertenence al bloque  0
Porcentaje de ser reasignado es: 0.5141395908543923
El producto  117 pertenence al bloque  0
Porcentaje de ser reasignado es: 0.5141395908543923
El producto  119 pertenence al bloque  0
Porcentaje de ser reasignado es: 0.5141395908543923
El producto  125 pertenence al bloque  0
Porcentaje de ser reasignado es: 0.5141395908543923
El producto  144 pertenence al bloque  0
Porcentaje de ser reasignado es: 0.5141395908543923
El producto  148 pertenence al bloque  0
Porcentaje de ser reasignado es: 0.5141395908543923
El producto  164 pertenence al bloque  0
Porcentaje de ser reasignado es: 0.5141395908543923
El producto  167 pertenence al bloque  0
Porcentaje de ser reasignado es: 0.5141395908543923
El producto  173 pertenence al bloque  0
Porcentaje de ser reasignado es: 0

,Customer_care_calls,Cost_of_the_Product,Discount_offered,Reached.on.Time_Y.N,Weight_in_gms
0,4.603845,237.925525,4.158527,1.387957e-01,1248.333660
1,4.143689,235.302123,1.000000,8.935073e-93,1525.890719
2,3.796285,189.797589,1.408393,1.519291e-64,4525.042419


,Customer_care_calls,Cost_of_the_Product,Discount_offered,Reached.on.Time_Y.N,Weight_in_gms
0,4.909941,239.293949,3.586093,1.187668e-01,1365.074161
1,4.823174,253.127595,1.000000,9.687425e-112,1192.148037
2,3.603081,212.249840,4.147084,0.000000e+00,4933.028910


Los cambios porcentuales en cada una de las variables son:



,0,1,2,3,4
0,0.066487,0.005751,1.376531e-01,0.144305,0.093517
1,0.163981,0.075756,2.591404e-07,1.000000,0.218720
2,0.050893,0.118296,1.944549e+00,1.000000,0.090162


In [13]:
#MÉTRICAS PARA EL CONJUNTO INTEGRADO

vars_num = ['Customer_care_calls','Cost_of_the_Product',
            'Discount_offered','Reached.on.Time_Y.N','Weight_in_gms']

# convertir XC2 a DataFrame
XC2_df = pd.DataFrame(XC2[:, :len(vars_num)], columns=vars_num)

def resumen_metricas(df, columnas):
    out = []
    for c in columnas:
        s = pd.to_numeric(df[c], errors='coerce').dropna()
        out.append({
            'Variable': c,
            'media': s.mean(),
            'desviacion': s.std(),
            'asimetria': skew(s, bias=False),
            'kurtosis': kurtosis(s, bias=False)
        })
    return pd.DataFrame(out)

metricas_global = resumen_metricas(XC2_df, vars_num)
display(metricas_global)


,Variable,media,desviacion,asimetria,kurtosis
0,Customer_care_calls,4.445399,0.730758,-1.704620,-1.5
1,Cost_of_the_Product,234.890461,20.791602,-0.910312,-1.5
2,Discount_offered,2.911059,1.678627,-1.516971,-1.5
3,Reached.on.Time_Y.N,0.039589,0.068570,1.732051,-1.5
4,Weight_in_gms,2496.750370,2111.649996,1.718993,-1.5


**Conclusión de resultados**

Los bloques originales del almacén tenían perfiles estadísticos variables en las métricas clave de interacción con los productos y los clientes. El bloque «E» no tenía datos disponibles para el análisis tras la limpieza inicial.
Esto se debe analizar a profundidad, ya que en este análisis se utilizó un método un poco más agresivo para la eliminación de datos Nulos u Outliers, pero despendiendo de lo que se quiera en cada caso, se puede trabajar con otro métodos menos agresivos como la imputación de datos, pero ya es una decisión que se toma teniendo en cuenta los requerimientos del proyecto y mirando la base de datos.

por otro lado, el proceso K-Medoids definió tres centros de clústeres distintos basándose en las características de los bloques iniciales A, B y C. Estos centros representan perfiles objetivo potenciales para los bloques integrados.
El intento de integrar los datos de otros bloques (C, D, E, F) dio lugar a cambios en estos centros de agrupación iniciales. Las métricas de estos centros modificados, proporcionan un resumen de alto nivel de las características medias dentro de los tres grupos tras esta integración parcial.

Al integrar los seis bloques el modelo consolidó la infromación sin perder consistencia en los datos. Si se analizan los resultados de las metricas globales, se identifica que las tres primeras variables (Customer_care_calls Cost_of_the_Product y Discount_offered) tuvieron una asímetri negativa, mientras que Reached.on.Time_Y.N	tuvo una asimetrÍa cercana a cero y Weight_in_gms fue la unica con una mejor asimetría (1,71). También podemos identificar que la Curtosis dio igual para todas las variables, esto puede estar relacionado a que se eliminaron todos los datos Nulos u Outliers, quedando así una distribucione plana, homogénea y sin valores extremos que pudieran distorsionar el análisis.

Se puede decir de este trabajo que el método seleccionado para integrar las variables cumplió a cabalidad, donde se garantizó datos estables sin outliers y se dio respuesta a la problematica incialmente planteada.
